# C1 BitFit

**Later illustrated companion; not the executed training notebook.** Historical metrics below are quoted from copied run records, never represented as notebook outputs.

The local dev score uses a threshold recorded in the run; inspect its calibration/selection record before treating it as independent model-selection evidence. A prepared model notebook is source scaffolding, while `v3_t4x2.ipynb` and the Kaggle run evidence identify the executed v3 lineage.

![Preserved computation graph](../figures/architecture.svg)

## Task and architecture

Only bias parameters in the C1 backbone receive gradients, with a trainable task evidence head. This tests an inexpensive adaptation of the same recurrent hybrid used by C1 full tuning.

**Adaptation:** Bias-only backbone tuning

Data path: official train → training-only fit/selection → official labeled dev report → 1,517-label-free test predictions. The original `code/` and `evidence/` directories contain the audit trail.

## Exact mechanism and experimental controls

### Provenance and scientific boundary

[CODE] C1: ten-layer encoder and four-layer GRU decoder with cross-attention

[CODE] For each unpadded chunk, the original backbone yields contextual token states. The head layer-normalizes them, then makes a softmax-weighted mixture of token mean, learned token attention and the last token. Normalized chunk positions enter through a learned projection. A bidirectional GRU contextualizes chunks in document order; learned attention and a chunk mean are mixed, then LayerNorm/dropout and a two-logit classifier predict the binary judgment.

[CODE/RUN] The backbone is frozen except bias parameters; the evidence pooling and two-class head are trainable.

### Exact recorded configuration

| Setting | Recorded value |
|---|---|
| Family | `C1: ten-layer encoder and four-layer GRU decoder with cross-attention` |
| Adaptation | `bitfit` |
| Train case cap | `12000` |
| Train-only holdout | `256 per class` |
| Chunking | `8 × 256 tokens; 8192 source characters` |
| Learning rates | `backbone 0.0002; head 0.0003` |
| Accumulation | `8` |
| Optimizer steps | `800` |
| Parameters | `118,799,242 total; 1,012,618 trainable` |
| Selected checkpoint | `4030ae989c2b84e8074c5ac2637ea877b84de4410dbdc4af173e80d1822c2378` |
| Data fingerprint | `88913c10c288a6f77a030d0391904d40b645fc242f2774f0ef8216e9f5afa792` |

### Optimization and selection

[CODE] The runner uses AdamW with separate backbone/head learning rates and weight decay 0.01. For each training example it computes class-weighted cross-entropy; weights are capped at 3.0. It periodically evaluates the fixed train-only holdout, saves the best checkpoint, and exports predictions from selected weights.

### Evaluation protocol

[RUN] A fixed, balanced training-only holdout of up to 512 cases selected the checkpoint and decision threshold. The official 994-case labeled dev set was evaluated after that selection; selected-threshold dev macro-F1 was 0.535560, while default 0.5 gave 0.454764. The hidden test labels were not available. These dev metrics are not the Codabench score shown in the user screenshot.

### Reproduction lineage

Initial six-job Kaggle v3 runner; the per-model notebook in code/ is prepared source, while v3_t4x2.ipynb and archived Kaggle outputs identify the executed run.

## Recorded result

Original Kaggle run: complete; 800 optimizer updates; selected step 400. Official 994-case dev macro-F1 **0.535560** at recorded threshold 0.449234; default 0.5 dev macro-F1 0.454764. Verified prediction ZIP SHA-256 `6d5be1e9fab3c623b9bb0323c54a5a95ab01f6421507a8fd9712e37c135d7c15`, exactly 1,517 IDs.

Local official dev, calibration CV, and Codabench scores are different measurements. A historical score is not an output of this notebook.

## Preserved run trace

![Recorded training loss or fitted stack coefficients](../figures/recorded_trace.svg)

The figure is drawn from copied run records, not from an execution of this companion notebook. The machine-readable source is in `evidence/`.

## Source excerpt: `models.py`

The complete source is in `code/`. This excerpt illustrates the preserved implementation; it is not a training replay.

In [ ]:
"""Four original backbones and six task adaptation strategies.

LoRMA uses an input-side multiplicative low-rank update. Adapters add a
bottleneck residual to FFN output projections. No external pretrained weights.
"""
from pathlib import Path
import torch
from torch import nn
from torch.nn import functional as F
from torch.utils.checkpoint import checkpoint
from phase3_rr.models import load_backbone,RRClassifier,adapt

class OutputAdapter(nn.Module):
 def __init__(self,base,bottleneck=32):
  super().__init__();self.base=base
  self.down=nn.Linear(base.out_features,bottleneck)
  self.up=nn.Linear(bottleneck,base.out_features)
  nn.init.zeros_(self.up.weight);nn.init.zeros_(self.up.bias)
 def forward(self,x):
  y=self.base(x);return y+self.up(F.gelu(self.down(y)))

def configure(backbone,family,method,rank=8,alpha=16):
 if method=='full':backbone.requires_grad_(True)
 elif method=='lorma':adapt(backbone,'lorma',rank,alpha)
 elif method=='bitfit':
  backbone.requires_grad_(False)
  for name,p in backbone.named_parameters():
   if name.rsplit('.',1)[-1].startswith('bias'):p.requires_grad_(True)
 elif method=='partial':
  if family!='b1':raise ValueError('This partial experiment is defined for B1')
  backbone.requires_grad_(False)
  for block in list(backbone.decoder)[-2:]+list(backbone.encoder)[-2:]:block.requires_grad_(True)
  backbone.decoder_norm.requires_grad_(True)
  if hasattr(backbone,'encoder_norm'):backbone.encoder_norm.requires_grad_(True)
 elif method=='adapters':
  backbone.requires_grad_(False)
  def walk(module):
   for name,child in list(module.named_children()):
    if name=='w2' and isinstance(child,nn.Linear):setattr(module,name,OutputAdapter(child))
    else:walk(child)
  walk(backbone)
 else:raise ValueError('Unknown adaptation method '+method)
 names=[n for n,p in backbone.named_parameters() if p.requires_grad]
 if not names:raise ValueError('No original-backbone/adaptation parameters trainable')
 return names

class CaseModel(nn.Module):
 def __init__(self,backbone,family,width):
  super().__init__();self.backbone=backbone;self.family=family;self.width=width
  self.pool=nn.Sequential(nn.Linear(width+2,64),nn.Tanh(),nn.Linear(64,1))
  self.head=nn.Sequential(nn.LayerNorm(width),nn.Dropout(.15),nn.Linear(width,2))
 def token_features(self,ids):return RRClassifier.representations(self,ids)
 def chunk_feature(self,ids):
  features=self.token_features(ids)
  # Chunks are unpadded, each encoded separately; all original tokens participate.
  return features.float().mean(1)
 def forward(self,chunks,positions,device):
  values=[]
  for ids in chunks:
   x=torch.as_tensor([ids],dtype=torch.long,device=device)
   if self.training:
    # Non-reentrant recomputation also preserves gradients with frozen embeddings.
    value=checkpoint(self.chunk_feature,x,use_reentrant=False)
   else:value=self.chunk_feature(x)
   values.append(value)
  h=torch.cat(values,0)
  pos=torch.as_tensor(positions,dtype=h.dtype,device=device).reshape(-1,1)
  scores=self.pool(torch.cat((h,pos,1-pos),-1)).float().squeeze(-1)
  weights=scores.softmax(0)
  return self.head((h*weights[:,None]).sum(0,keepdim=True))

class EvidenceCaseModel(CaseModel):
 """Learned token evidence and ordered chunk context; own backbone stays intact."""
 def __init__(self,backbone,family,width):
  super().__init__(backbone,family,width)
  hidden=min(128,max(4,width//2))
  self.token_norm=nn.LayerNorm(width)
  self.token_score=nn.Sequential(nn.Linear(width,min(64,width)),nn.Tanh(),nn.Linear(min(64,width),1))
  nn.init.zeros_(self.token_score[-1].weight);nn.init.zeros_(self.token_score[-1].bias)
  self.token_mix=nn.Parameter(torch.zeros(3))
  self.position=nn.Linear(2,width,bias=False);nn.init.zeros_(self.position.weight)
  self.chunk_context=nn.GRU(width,hidden,bidirectional=True,batch_first=True)
  self.context_projection=nn.Linear(2*hidden,width)
  self.context_scale=nn.Parameter(torch.tensor(.1))
  self.context_norm=nn.LayerNorm(width)
  self.document_mix=nn.Parameter(torch.zeros(2))
 def chunk_feature(self,ids):
  values=self.token_features(ids).float()
  values=values[:,1:-1] if values.shape[1]>2 else values
  values=self.token_norm(values)
  attention=self.token_score(values).float().squeeze(-1).softmax(-1)
  summaries=torch.stack((values.mean(1),(values*attention[:,:,None]).sum(1),values[:,-1]),0)
  return (summaries*self.token_mix.softmax(0)[:,None,None]).sum(0)
 def forward(self,chunks,positions,device):
  if not chunks or len(chunks)!=len(positions):raise ValueError('Chunk/position mismatch')
  values=[]
  for ids in chunks:
   x=torch.as_tensor([ids],dtype=torch.long,device=device)
   values.append(checkpoint(self.chunk_feature,x,use_reentrant=False) if self.training else self.chunk_feature(x))
  h=torch.cat(values,0).float()
  pos=torch.as_tensor(positions,dtype=h.dtype,device=device).reshape(-1,1)
  h=h+self.position(torch.cat((pos,1-pos),-1))
  with torch.autocast(device_type=device.type,enabled=False):
   context,_=self.chunk_context(h[None].float())
   h=self.context_norm(h+self.context_scale*self.context_projection(context[0]))
  attention=self.pool(torch.cat((h,pos,1-pos),-1)).float().squeeze(-1).softmax(0)
  summaries=torch.stack(((h*attention[:,None]).sum(0),h.mean(0)),0)
  document=(summaries*self.document_mix.softmax(0)[:,None]).sum(0,keepdim=True)
  return self.head(document)


## Verify the copied run evidence

The next cell is a read-only inspection helper. It does not train or generate new scores. Running it is optional; this delivered notebook has no saved execution output.

In [ ]:
from pathlib import Path
import json
root = Path('..')
for record in sorted((root / 'evidence').glob('*terminal.json')):
    result = json.loads(record.read_text(encoding='utf-8'))
    print(record.name, result.get('name'), result.get('state'), result.get('dev_metrics', {}).get('macro_f1'))


## Reproduction boundary

The package has code, configs, selected records, and prediction ZIP. Original model weights and official data are retained in the local archives, not duplicated here. Paths in the original configs must be remapped before an authorized rerun. The source code can be reviewed without executing it.